<a href="https://colab.research.google.com/github/RegmiYogesh/Object_Centric_Patch-_sampling/blob/main/Multi_class_cotton_Water.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Multiclass Patch Generation for Cotton and Water

This section adapts the previous patch generation logic to create datasets for a multiclass segmentation model. We will define distinct class IDs for 'cotton' and 'water', combine their geometries, and generate label masks that can contain both classes.

In [3]:
import os
import pandas as pd # Explicitly import pandas for concat
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
import numpy as np
from shapely.geometry import Point, box
import random

# ==============================
# PATHS (re-defined for self-contained execution)
# ==============================
satellite_fp = r"D:\Paper\Cotton_Fnal.tif"
cotton = r"D:\Paper\Whole_cotton.shp"
water = r"D:\Paper\water.shp"
base_output = r'D:\Paper'

PATCH_SIZE = 256
HALF = PATCH_SIZE // 2
BACKGROUND_RATIO = 0.20

# ==============================
# MULTICLASS CONFIGURATION
# ==============================
COTTON_CLASS = 1
WATER_CLASS = 2
# Background is implicitly 0

# Define new output directories for multiclass images and labels
multi_img_dir = os.path.join(base_output, "images_multiclass")
multi_lbl_dir = os.path.join(base_output, "labels_multiclass")

os.makedirs(multi_img_dir, exist_ok=True)
os.makedirs(multi_lbl_dir, exist_ok=True)

# ==============================
# LOAD VECTOR DATA FOR MULTICLASS
# ==============================
# Load cotton and assign class ID
cotton_gdf = gpd.read_file(cotton)
cotton_gdf['class_id'] = COTTON_CLASS

# Load water and assign class ID
water_gdf = gpd.read_file(water)
water_gdf['class_id'] = WATER_CLASS

# Combine all feature GeoDataFrames
# It's important to keep track of geometry, class_id, and any other relevant columns
all_features_gdf = pd.concat([cotton_gdf, water_gdf], ignore_index=True)


with rasterio.open(satellite_fp) as src:

    # Reproject combined vector data to raster CRS
    all_features_gdf = all_features_gdf.to_crs(src.crs)

    # ---------- METADATA ----------
    # Copy img_meta from src to match existing pattern
    img_meta = src.meta.copy()

    multi_lbl_meta = src.meta.copy()
    multi_lbl_meta.update(
        {
            "count": 1,
            "dtype": "uint8", # Ensure dtype can hold 0, 1, 2
            # Removed "nodata": 0 to ensure 0 values are not masked out
        }
    )

    # ---------- UTILITY FUNCTION (re-defined in this scope for clarity and independence) ----------
    def is_window_valid(w):
        return (
            w.col_off >= 0
            and w.row_off >= 0
            and w.col_off + w.width <= src.width
            and w.row_off + w.height <= src.height
        )

    saved_multi_feature_count = 0

    # ==============================
    # 1‴ MULTICLASS FEATURE PATCHES (COTTON & WATER)
    # ==============================
    for idx, row in all_features_gdf.iterrows():

        class_id_val = row['class_id']
        feature_name = 'cotton' if class_id_val == COTTON_CLASS else 'water' # For file naming

        centroid = row.geometry.centroid

        # Map → pixel
        col, row_pix = ~src.transform * (centroid.x, centroid.y)
        col, row_pix = int(col), int(row_pix)

        window = Window(col - HALF, row_pix - HALF, PATCH_SIZE, PATCH_SIZE)

        if not is_window_valid(window):
            continue

        # ---------- IMAGE ----------
        image = src.read(window=window)
        win_transform = src.window_transform(window)

        img_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        img_fp = os.path.join(multi_img_dir, f"{feature_name}_{idx}.tif")
        with rasterio.open(img_fp, "w", **img_meta) as dst:
            dst.write(image)

        # ---------- LABEL (MULTICLASS) ----------
        label = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)

        window_bounds = rasterio.windows.bounds(window, src.transform)
        window_geom = box(*window_bounds)

        # Get all features that intersect with the current window
        intersecting_features_in_window = all_features_gdf[all_features_gdf.intersects(window_geom)]

        if not intersecting_features_in_window.empty:
            # Create (geometry, value) pairs for rasterize
            geometries_to_burn = [
                (g, c_id) for g, c_id in zip(intersecting_features_in_window.geometry, intersecting_features_in_window['class_id'])
            ]
            burned = rasterize(
                geometries_to_burn,
                out_shape=(PATCH_SIZE, PATCH_SIZE),
                transform=win_transform,
                fill=0, # Fill background with 0
                dtype="uint8",
            )
            label = np.maximum(label, burned) # Use np.maximum to handle overlaps and prioritize higher class IDs

        multi_lbl_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        lbl_fp = os.path.join(multi_lbl_dir, f"{feature_name}_{idx}.tif")
        with rasterio.open(lbl_fp, "w", **multi_lbl_meta) as dst:
            dst.write(label, 1)

        saved_multi_feature_count += 1

    print(f"Saved {saved_multi_feature_count} multiclass feature image-label pairs (cotton and water)")

    # ==============================
    # 2‴ BACKGROUND PATCHES FOR MULTICLASS
    # ==============================
    # Calculate desired number of background patches based on the new feature count
    num_background_multi = int(saved_multi_feature_count * BACKGROUND_RATIO)

    bg_multi_count = 0
    attempts_multi = 0

    while bg_multi_count < num_background_multi and attempts_multi < num_background_multi * 10:

        attempts_multi += 1

        col = random.randint(HALF, src.width - HALF)
        row_pix = random.randint(HALF, src.height - HALF)

        x, y = src.transform * (col, row_pix)
        point = Point(x, y)

        # Skip if centroid is inside any of the combined features (cotton or water)
        if all_features_gdf.contains(point).any():
            continue

        window = Window(col - HALF, row_pix - HALF, PATCH_SIZE, PATCH_SIZE)

        if not is_window_valid(window):
            continue

        # Check if the candidate background window intersects ANY of the combined features
        window_bounds = rasterio.windows.bounds(window, src.transform)
        window_geom = box(*window_bounds)
        intersecting_any_features = all_features_gdf[all_features_gdf.intersects(window_geom)]
        if not intersecting_any_features.empty:
            continue  # This patch contains features, so it's not a true background patch

        # ---------- IMAGE ----------
        image = src.read(window=window)

        # Skip if the image is completely black (no data)
        if np.all(image == 0):
            continue

        win_transform = src.window_transform(window)

        img_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        img_fp = os.path.join(multi_img_dir, f"background_multiclass_{bg_multi_count}.tif")
        with rasterio.open(img_fp, "w", **img_meta) as dst:
            dst.write(image)

        # ---------- LABEL (ALL ZERO FOR BACKGROUND) ----------
        label = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)

        multi_lbl_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        lbl_fp = os.path.join(multi_lbl_dir, f"background_multiclass_{bg_multi_count}.tif")
        with rasterio.open(lbl_fp, "w", **multi_lbl_meta) as dst:
            dst.write(label, 1)

        bg_multi_count += 1

    print(f"Saved {bg_multi_count} multiclass background image-label pairs")

Saved 3371 multiclass feature image-label pairs (cotton and water)
Saved 674 multiclass background image-label pairs
